In [1]:
import pandas as pd
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
import time
from time import sleep
load_dotenv()
API_KEY = os.getenv("API_KEY")

In [ ]:
df1 = pd.read_csv("authors_data.csv")

indexes = df1.index[
    (df1["works_count"] >= 5) &
    (df1["works_count"] <= 5000)
].tolist()

subset = df1.loc[indexes, "author_id"]
print(len(subset))

1333


In [ ]:

import requests
import pandas as pd
from itertools import islice
from time import sleep

# Sociology, Psychology, Economics, Political Science
css_concepts = ["C157449823", "C77805123", "C162324750", "C17744445"]
# Mathematics, Physics, Computer Science
quant_concepts = ["C33923547", "C121332964", "C41008148"]

# Build the concept filter string
# Works must have ANY CSS concept AND ANY quant concept
concept_filter1 = f"concept.id:{'|'.join(css_concepts)}"
concept_filter2 = f"concept.id:{'|'.join(quant_concepts)}"

BASE_URL = "https://api.openalex.org/works"

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

rows2 = []
rows3 = []
seen_work_ids = set()

for batch_num, author_batch in enumerate(chunked(subset, 25), start=1):

    author_filter = "author.id:" + "|".join(author_batch)

    filter_string = (
    f"{author_filter},"
    f"cited_by_count:>10,"
    f"authors_count:<10,"
    f"{concept_filter1},"
    f"{concept_filter2}"
)

    cursor = "*"

    print(f"\nProcessing batch {batch_num}")

    while cursor:

        params = {
            "filter": filter_string,
            "select": "publication_year,id,cited_by_count,corresponding_author_ids,title,abstract_inverted_index",
            "per-page": 200,
            "cursor": cursor,
            "api_key": API_KEY
        }

        response = requests.get(BASE_URL, params=params)
        data = response.json()

        results = data.get("results", [])
        if not results:
            break

        print(f"  Fetched {len(results)} works")

        for item in results:
            work_id = item.get("id")

            # Deduplicate across batches
            if work_id in seen_work_ids:
                continue
            seen_work_ids.add(work_id)

            rows2.append({
                "id": work_id,
                "publication_year": item.get("publication_year"),
                "cited_by_count": item.get("cited_by_count"),
                "corresponding_author_ids": item.get("corresponding_author_ids"),
            })

            rows3.append({
                "id": work_id,
                "title": item.get("title"),
                "abstract_index": item.get("abstract_inverted_index"),
            })

        cursor = data["meta"].get("next_cursor")

        sleep(0.1)

    sleep(0.5)

df2 = pd.DataFrame(rows2)
df3 = pd.DataFrame(rows3)

print("\nComplete!")
len(df2), len(df3)


Processing batch 1
  Fetched 200 works
  Fetched 137 works

Processing batch 2
  Fetched 200 works
  Fetched 164 works

Processing batch 3
  Fetched 200 works
  Fetched 105 works

Processing batch 4
  Fetched 152 works

Processing batch 5
  Fetched 200 works
  Fetched 114 works

Processing batch 6
  Fetched 200 works
  Fetched 153 works

Processing batch 7
  Fetched 200 works
  Fetched 19 works

Processing batch 8
  Fetched 200 works
  Fetched 200 works
  Fetched 51 works

Processing batch 9
  Fetched 200 works
  Fetched 132 works

Processing batch 10
  Fetched 171 works

Processing batch 11
  Fetched 200 works
  Fetched 29 works

Processing batch 12
  Fetched 139 works

Processing batch 13
  Fetched 184 works

Processing batch 14
  Fetched 151 works

Processing batch 15
  Fetched 133 works

Processing batch 16
  Fetched 200 works
  Fetched 29 works

Processing batch 17
  Fetched 157 works

Processing batch 18
  Fetched 200 works
  Fetched 180 works

Processing batch 19
  Fetched 200 

(11706, 11706)

In [4]:
df2.to_csv("works_ids.csv", index=False)
df3.to_csv("works_titles_abstracts.csv", index=False)

## Co-Authors)

In [8]:
co_author_ids = set()
corresponding_unique_author_ids = df2["corresponding_author_ids"].explode().unique()
for author_id in corresponding_unique_author_ids:
    if author_id not in df1["author_id"].values:
        co_author_ids.add(author_id)

len(co_author_ids)

5532

In [ ]:
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = "https://api.openalex.org/authors/"
MAX_WORKERS = 4   # keep low to avoid 429

session = requests.Session()  # reuse TCP connections

def fetch_author(author_id):
    short_id = author_id.split("/")[-1]
    url = BASE_URL + short_id

    params = {
        "select": "display_name,works_count,works_api_url,last_known_institutions,summary_stats",
        "api_key": API_KEY
    }

    for attempt in range(5):
        try:
            response = session.get(url, params=params, timeout=10)

            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 2))
                time.sleep(wait)
                continue

            response.raise_for_status()
            data = response.json()

            h_index = data.get("summary_stats", {}).get("h_index")
            institutions = data.get("last_known_institutions", [])
            country_code = institutions[0].get("country_code") if institutions else None

            return {
                "display_name": data.get("display_name"),
                "works_count": data.get("works_count"),
                "works_api_url": data.get("works_api_url"),
                "h_index": h_index,
                "country_code": country_code
            }

        except requests.exceptions.RequestException:
            time.sleep(1)

    return None


rows4 = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_author, aid) for aid in co_author_ids]

    for future in as_completed(futures):
        result = future.result()
        if result:
            rows4.append(result)

df4 = pd.DataFrame(rows4)
df4.to_csv("co_authors_data.csv", index=False)

print("Done. Authors fetched:", len(df4))

Done. Authors fetched: 5532


In [ ]:
# import requests
# import pandas as pd
# from concurrent.futures import ThreadPoolExecutor

# def fetch_author(author_id):
#     URL = "https://api.openalex.org/authors"
#     params = {
#         "filter": f"author.id:{author_id}",
#         "select": "display_name,works_count,works_api_url,last_known_institutions,summary_stats",
#         "api_key": API_KEY
#     }
#     try:
#         response = requests.get(URL, params=params, timeout=10)
#         response.raise_for_status()
#         time.sleep(0.2)  # add a small delay
#         data = response.json()
#     except requests.exceptions.RequestException as e:
#         print(f"Request failed for {author_id}: {e}")
#         return None

#     if data.get("results"):
#         author_data = data["results"][0]
#         h_index = author_data.get("summary_stats", {}).get("h_index")
#         institutions = author_data.get("last_known_institutions", [])
#         country_code = institutions[0].get("country_code") if institutions else None

#         # Return the author dictionary
#         return {
#             "display_name": author_data.get("display_name"),
#             "works_count": author_data.get("works_count"),
#             "works_api_url": author_data.get("works_api_url"),
#             "h_index": h_index,
#             "country_code": country_code
#         }
#     else:
#         print(f"No results found for author: {author_id}")
#         return None

# # Parallel execution
# with ThreadPoolExecutor(max_workers=1) as executor:
#     results = executor.map(fetch_author, co_author_ids)

# # Only keep the results that are not None
# rows4 = [r for r in results if r]

# # Create DataFrame
# df4 = pd.DataFrame(rows4)
# df4.to_csv("co_authors_data.csv", index=False)
# print("Done! Authors saved:", len(df4))

## Co-author Works

In [1]:
rows5 = []
rows6 = []


# Sociology, Psychology, Economics, Political Science
css_concepts = ["C157449823", "C77805123", "C162324750", "C17744445"]
# Mathematics, Physics, Computer Science
quant_concepts = ["C33923547", "C121332964", "C41008148"]

# Build the concept filter string
# Works must have ANY CSS concept AND ANY quant concept
concept_filter1 = f"concept.id:{'|'.join(css_concepts)}"
concept_filter2 = f"concept.id:{'|'.join(quant_concepts)}"

BASE_URL = "https://api.openalex.org/works"

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

seen_work_ids = set()

for batch_num, author_batch in enumerate(chunked(co_author_ids, 25), start=1):

    author_filter = "author.id:" + "|".join(author_batch)

    filter_string = (
    f"{author_filter},"
    f"cited_by_count:>10,"
    f"authors_count:<10,"
    f"{concept_filter1},"
    f"{concept_filter2}"
)

    cursor = "*"

    print(f"\nProcessing batch {batch_num}")

    while cursor:

        params = {
            "filter": filter_string,
            "select": "publication_year,id,cited_by_count,corresponding_author_ids,title,abstract_inverted_index",
            "per-page": 200,
            "cursor": cursor,
            "api_key": API_KEY
        }

        response = requests.get(BASE_URL, params=params)
        data = response.json()
        print(data)

        results = data["results"]
        for work_id in results[0].get("id"):
            if work_id in df2["id"].values:
                break
        else:
            pass
        only_co_and_author = []
        corresponding_author_ids = item.get("corresponding_author_ids")
        for corresponding_author_id in corresponding_author_ids:
            if corresponding_author_id in co_author_ids:
                only_co_and_author.append(corresponding_author_id)
            elif corresponding_author_id in subset:
                only_co_and_author.append(corresponding_author_id)

        print(f"  Fetched {len(results)} works")

        for item in results:
            work_id = item.get("id")

            # Deduplicate across batches
            if work_id in seen_work_ids:
                continue
            seen_work_ids.add(work_id)

            rows5.append({
                "id": work_id,
                "publication_year": item.get("publication_year"),
                "cited_by_count": item.get("cited_by_count"),
                "corresponding_author_ids": only_co_and_author,
            })

            rows6.append({
                "id": work_id,
                "title": item.get("title"),
                "abstract_index": item.get("abstract_inverted_index"),
            })

        cursor = data["meta"].get("next_cursor")

        sleep(0.1)

    sleep(0.5)

df5 = pd.DataFrame(rows5)
df6 = pd.DataFrame(rows6)

print("\nComplete!")
len(df5), len(df6)


NameError: name 'co_author_ids' is not defined

In [ ]:
df5.to_csv("co_authors_works_ids.csv", index=False)
df6.to_csv("co_authors_works_titles_abstracts.csv", index=False)